In [ ]:
# Example 1: public API surface — confirm the documented exports exist and are callable.
import griffecli

expected = {"DEFAULT_LOG_LEVEL", "check", "dump", "get_parser", "main"}
assert expected.issubset(set(griffecli.__all__)), (
    f"Missing exports: {expected - set(griffecli.__all__)}"
)

for name in ("check", "dump", "get_parser", "main"):
    obj = getattr(griffecli, name)
    assert callable(obj), f"griffecli.{name} is not callable: {obj!r}"
    print(f"griffecli.{name}: {obj!r}")

assert isinstance(griffecli.DEFAULT_LOG_LEVEL, str)
print(f"griffecli.DEFAULT_LOG_LEVEL = {griffecli.DEFAULT_LOG_LEVEL!r}")


In [ ]:
# Example 2: get_parser() returns a configured argparse.ArgumentParser with subcommands.
import argparse

from griffecli import get_parser

parser = get_parser()
assert isinstance(parser, argparse.ArgumentParser)
print(f"parser.prog = {parser.prog!r}")

# Collect subparser names — griffecli exposes 'check' and 'dump' subcommands.
subparsers_actions = [
    a for a in parser._actions if isinstance(a, argparse._SubParsersAction)
]
assert subparsers_actions, "expected a subparsers action on the griffecli parser"
subcommand_names = set(subparsers_actions[0].choices)
print(f"subcommands = {sorted(subcommand_names)}")
assert {"check", "dump"}.issubset(subcommand_names), (
    f"missing expected subcommands: {subcommand_names}"
)


In [ ]:
# Example 3: `griffecli --help` exits cleanly via SystemExit with code 0.
from griffecli import main

try:
    rc = main(["--help"])
except SystemExit as e:
    code = e.code if e.code is not None else 0
    print(f"main(['--help']) raised SystemExit({code})")
    assert code == 0, f"--help should exit 0, got {code}"
else:
    print(f"main(['--help']) returned {rc}")
    assert rc == 0, f"--help should return 0, got {rc}"


In [ ]:
# Example 4: parse a 'dump' invocation without executing it — verify argument wiring.
from griffecli import get_parser

parser = get_parser()
ns = parser.parse_args(["dump", "some_pkg", "-o", "/tmp/out.json"])
print(f"subcommand: {ns.subcommand}")
print(f"packages: {ns.packages}")
print(f"output: {ns.output}")

assert ns.subcommand == "dump"
assert ns.packages == ["some_pkg"]
assert ns.output == "/tmp/out.json"


In [ ]:
# Example 5: invoking griffecli with no arguments prints help and returns non-zero.
import io
from contextlib import redirect_stderr, redirect_stdout

from griffecli import main

stdout_buf, stderr_buf = io.StringIO(), io.StringIO()
try:
    with redirect_stdout(stdout_buf), redirect_stderr(stderr_buf):
        rc = main([])
except SystemExit as e:
    rc = e.code if e.code is not None else 0

combined = (stdout_buf.getvalue() + stderr_buf.getvalue()).lower()
print(f"main([]) -> {rc}")
print(f"output snippet: {combined[:200]!r}")

# Should have produced help-like output and a non-zero return (no subcommand chosen).
assert rc != 0, f"expected non-zero rc when no subcommand is given, got {rc}"
assert "usage" in combined or "griffe" in combined, (
    "expected help-style output to mention usage/griffe"
)
